In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

In [22]:
df = pd.read_csv("Diabetes_and_LifeStyle_Dataset_.csv")

print(df.shape)
df.head()

(97297, 31)


,Age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
1,52,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,...,55,50,30,93,150,2.00,5.63,23.0,No Diabetes,0
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,...,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,...,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,...,52,125,160,137,184,12.74,7.20,23.5,Type 2,1


In [23]:
target_col = "diabetes_stage"

In [24]:
X = df.drop(columns=[target_col])
y = df[target_col]

In [25]:
leak_cols = [
    "diagnosed_diabetes",
    "diabetes_risk_score"
]
X = X.drop(columns=leak_cols, errors="ignore")

In [26]:
X = pd.get_dummies(X, drop_first=True)

In [27]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [29]:
dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=5,
    random_state=42
)

dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

Decision Tree Accuracy: 0.9161870503597123


In [30]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Accuracy: 0.9118191161356629


In [31]:
xgb = XGBClassifier(
    eval_metric='mlogloss',
    random_state=42
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))

XGBoost Accuracy: 0.9157245632065776


In [32]:
feature_importance = pd.Series(rf.feature_importances_, index=X.columns)
feature_importance.sort_values(ascending=False).head(10)

hba1c                                 0.558211
glucose_postprandial                  0.234857
glucose_fasting                       0.162371
family_history_diabetes               0.014683
Age                                   0.005546
physical_activity_minutes_per_week    0.003818
bmi                                   0.002624
systolic_bp                           0.002084
ldl_cholesterol                       0.001463
triglycerides                         0.001452
dtype: float64

In [33]:
feature_importance.sort_values(ascending=False).to_csv(
    "../artifacts/feature_importance.csv"
)

In [34]:
cluster_df = df.drop([
    "diagnosed_diabetes",
    "diabetes_risk_score",
    "hba1c",
    "glucose_fasting",
    "glucose_postprandial",
    "insulin_level"
], axis=1)

cluster_df = pd.get_dummies(cluster_df, drop_first=True)

In [35]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(cluster_df)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(scaled_features)

df["Cluster"] = cluster_labels

df["Cluster"].value_counts()

Cluster
1    34279
0    33475
2    29543
Name: count, dtype: int64

In [ ]:
sil_score = silhouette_score(scaled_features, cluster_labels)
print("Silhouette Score:", sil_score)

In [ ]:
cluster_profiles = df.groupby("Cluster").mean(numeric_only=True)
cluster_profiles.T.head(15)

In [ ]:
print("\n=== FINAL INSIGHTS ===")
print("""
- Random Forest and XGBoost generally perform best for classification.
- Lifestyle factors such as BMI, activity, and diet influence diabetes risk.
- Clustering reveals 3 patient groups: low-risk, moderate-risk, and high-risk.
- Silhouette score indicates how well-separated the clusters are.
""")


print("\n=== RECOMMENDATIONS ===")
print("""
- Encourage physical activity and healthy diets.
- Monitor high-risk individuals more closely.
- Use predictive models in clinical decision support systems.
""")